In [32]:
import pandas as pd
df=pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [33]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [34]:
import re
negation_words=[
    'not good','not bad','not great',"don't like","didn't like",'never liked',"wasn't good","isn't good",'no good'
    ]

In [35]:
def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z\s']"," ",text)
  #convert negations into single tokens
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [36]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    df['review'],df['sentiment'],test_size=0.2,random_state=42
)

In [37]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size=20000
max_len=250

tokenizer=Tokenizer(num_words=vocab_size,oov_token="<oov>")#oov -> out of vocabulary
tokenizer.fit_on_texts(x_train)

x_train_seq=tokenizer.texts_to_sequences(x_train)
x_test_seq=tokenizer.texts_to_sequences(x_test)#convert tokens into sequences eg:-[0,1,2...]. it is required to process using lstm,gru

x_train_pad=pad_sequences(x_train_seq,maxlen=max_len,padding='post')
x_test_pad=pad_sequences(x_test_seq,maxlen=max_len,padding='post')

In [38]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout
model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [39]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [40]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [41]:
history=model.fit(x_train_pad,y_train,batch_size=64,epochs=5,validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 437s 863ms/step - accuracy: 0.5522 - loss: 0.6693 - val_accuracy: 0.5803 - val_loss: 0.6284
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 433s 866ms/step - accuracy: 0.6031 - loss: 0.6096 - val_accuracy: 0.5850 - val_loss: 0.6246
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 430s 861ms/step - accuracy: 0.7025 - loss: 0.5362 - val_accuracy: 0.7416 - val_loss: 0.5627
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 439s 879ms/step - accuracy: 0.7864 - loss: 0.4716 - val_accuracy: 0.8087 - val_loss: 0.5086
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 434s 868ms/step - accuracy: 0.8405 - loss: 0.3821 - val_accuracy: 0.8411 - val_loss: 0.3975


In [42]:
loss,acc=model.evaluate(x_test_pad,y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 34s 106ms/step - accuracy: 0.8404 - loss: 0.4042
Test Accuracy: 0.840399980545044


In [43]:
def predict_sentiment(review):
  review=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]
  print("\nReview:",review)
  print("Score:",prediction)
  if prediction>=0.5:
    print("Sentiment: Positive ")
  else:
    print("Sentiment: Negative ")

In [45]:
predict_sentiment("Tis movie was absolutely amazing and i loved it very much")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 593ms/step

Review: tis movie was absolutely amazing and i loved it very much
Score: 0.8895934
Sentiment: Positive 


In [47]:
predict_sentiment("this is a bad movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step

Review: this is a bad movie
Score: 0.22068748
Sentiment: Negative 


In [48]:
predict_sentiment("i didn't lik this movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step

Review: i didn't lik this movie
Score: 0.43670171
Sentiment: Negative 
